# [GALAXY] – SIMBAD Downloader

<div class="alert alert-block alert-info">
<b>Environment:</b> Run this notebook in the <code>stenv</code> conda environment.
</div>

## Imports

In [ ]:
# Python Imports
import os
from pathlib import Path

# Astropy Collaboration Imports
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import QTable, Table
from astroquery.simbad import Simbad
from regions import PointSkyRegion, Regions

## Notebook Setup

In [ ]:
if Path.cwd().name != "SIMBAD":
    if Path.cwd().name == "Notebooks":
        os.chdir("../Data/SIMBAD")
    else:
        raise RuntimeError(
            "This notebook must be run from the SIMBAD directory."
        )
print(f'Current Directory: {Path.cwd()}')

In [ ]:
# Input data
NED_DATA_FILE = Path('../NED/[GALAXY_SHORT]-NED_Data.ecsv')

# Query parameters
FOV_RADIUS_ARCMIN = 20.0  # arcminutes — adjust to match image FOV

# Output file paths
OUT_CATALOG_ECSV = Path('[GALAXY_SHORT]-SIMBAD-AlignmentStars.ecsv')
OUT_COORDS_ECSV = Path('[GALAXY_SHORT]-SIMBAD-AlignmentStars-Coordinates.ecsv')
OUT_COORDS_REG = Path('[GALAXY_SHORT]-SIMBAD-AlignmentStars-Coordinates.reg')
OUT_TWEAKREG_CAT = Path('[GALAXY_SHORT]-SIMBAD-RefCatalog-icrs.txt')

## Load External Data

Load the NED galaxy data table to obtain the galaxy's sky coordinates,
which define the centre of the SIMBAD search cone.

In [ ]:
ned_data_table = Table.read(NED_DATA_FILE)
gal_crd = SkyCoord(
    ra=ned_data_table['RA'][0],
    dec=ned_data_table['DEC'][0],
    unit='deg',
    frame='fk5'
)
print(f'Galaxy Coordinates (FK5): {gal_crd}')
print(f'Galaxy Coordinates (ICRS): {gal_crd.icrs}')

## Query SIMBAD

Query the SIMBAD database for all objects within `FOV_RADIUS_ARCMIN` arcminutes
of the galaxy centre.  The `otype` votable field is requested so that results
can be filtered to stellar point sources suitable for TweakReg alignment.

In [ ]:
# Configure SIMBAD to return the object type
simbad = Simbad()
simbad.add_votable_fields('otype')

# Run the cone search
obj_table = simbad.query_region(
    gal_crd.icrs,
    radius=FOV_RADIUS_ARCMIN * u.arcmin
)

print(f'Found {len(obj_table):d} total objects in the FOV')

## Filter to Stellar Sources

TweakReg requires unresolved point sources for alignment.  SIMBAD object types
that begin with `'*'` represent stars in the SIMBAD type hierarchy.  All other
types (galaxies, clusters, nebulae, etc.) are excluded.  A summary of the
type breakdown is printed before and after filtering.

In [ ]:
# Print object-type breakdown before filtering
print('Object type breakdown (all sources):')
for otype in sorted(set(str(t) for t in obj_table['OTYPE'])):
    n = sum(1 for t in obj_table['OTYPE'] if str(t) == otype)
    print(f'  {otype:<20s} {n:>4d}')

In [ ]:
# Keep only stellar objects (SIMBAD otype codes that start with '*')
star_mask = [str(otype).startswith('*') for otype in obj_table['OTYPE']]
star_table = obj_table[star_mask]

print(f'Retained {len(star_table):d} stellar objects for alignment')

In [ ]:
# Build SkyCoord array from the filtered table.
# SIMBAD returns RA as a sexagesimal string ("HH MM SS.sss") and
# DEC as a sexagesimal string ("DD MM SS.ss") by default.
obj_coords = SkyCoord(
    ra=star_table['RA'],
    dec=star_table['DEC'],
    unit=(u.hourangle, u.deg),
    frame='icrs'
)

print(f'Coordinate array built: {len(obj_coords):d} entries')

## Write Outputs

Write the filtered stellar catalogue to four output files:

1. **ECSV catalogue** — full `star_table` with SIMBAD metadata.
2. **Coordinates ECSV** — `SkyCoord` column only, for convenient re-loading.
3. **DS9 region file** — point markers for visual inspection.
4. **TweakReg plain-text catalogue** — whitespace-separated RA Dec (ICRS decimal
   degrees) as required by `TweakReg`.

In [ ]:
# 1. Write the full filtered table
star_table.write(OUT_CATALOG_ECSV, overwrite=True)
print(f'Wrote full catalogue  -> {OUT_CATALOG_ECSV}')

In [ ]:
# 2. Write SkyCoord-only ECSV
# Reload with: QTable.read(OUT_COORDS_ECSV)['SkyCoord']
QTable([obj_coords], names=['SkyCoord']).write(
    OUT_COORDS_ECSV, overwrite=True
)
print(f'Wrote coordinates     -> {OUT_COORDS_ECSV}')

In [ ]:
# 3. Write DS9 region file with cyan 'x' markers
regs = Regions([])
for crd in obj_coords:
    regs.append(PointSkyRegion(crd))
regs.write(OUT_COORDS_REG, overwrite=True)

# Insert global style directive after the first line
with open(OUT_COORDS_REG, 'r') as fid:
    lines = fid.readlines()
lines.insert(1, 'global point=x color=cyan\n')
with open(OUT_COORDS_REG, 'w') as fid:
    fid.writelines(lines)

print(f'Wrote DS9 regions     -> {OUT_COORDS_REG}')

In [ ]:
# 4. Write TweakReg plain-text catalogue (whitespace-separated RA Dec)
# TweakReg requires: 'text files containing whitespace-separated list of values'
with open(OUT_TWEAKREG_CAT, 'w') as fid:
    for crd in obj_coords:
        fid.write(
            f'{crd.icrs.ra.value:<20.8f} {crd.icrs.dec.value:<20.8f}\n'
        )

print(f'Wrote TweakReg cat.   -> {OUT_TWEAKREG_CAT}')
print(f'Total alignment stars: {len(obj_coords):d}')